#Assignment #6

Vision Transformer

---


**SUBMISSION INSTRUCTIONS**

First make a copy of this colab file and then solve the assignment and upload your final notebook on github.

Before uploading your downloaded notebook, RENAME the file as rollno_name.ipynb

Submission Deadline : 31/01/2026 Saturday EOD i.e before 11:59 PM

The deadline is strict, Late submissions will incur penalty

Note that you have to upload your solution on the github page of the project Vision Transformer and **under Assignment 6**

And remember to keep title of your pull request to be ViT_name_rollno_assgn6

Github Submission repo -
https://github.com/electricalengineersiitk/Winter-projects-25-26/tree/main/Vision%20transformer/Assignment%206

# Part A - Data and tokens

Q1. Build a tiny toy dataset with pandas
Create a pandas DataFrame with columns text and label.
- Include at least 12 short sentences (3-10 words each).
- The label is 0/1 (e.g., positive vs negative sentiment).
- Shuffle rows and split into train/test (80/20) using a fixed random seed.
Return: df_train, df_test.

In [40]:
import pandas as pd
from sklearn.model_selection import train_test_split

def make_toy_dataset(seed: int = 42):
    raw_data = {
        "text": [
            "The aerodynamics of the wing are perfect.",
            "I love the campus life at IITK.",
            "Hardware security is extremely difficult to master.",
            "The food in the canteen is terrible today.",
            "The flight controller is malfunctioning and unstable.",
            "This vision transformer implementation is very clean.",
            "The weather in Kanpur is way too dusty.",
            "Our team won the robotics competition easily.",
            "The lecture on fluid mechanics was very boring.",
            "The propulsion system has high efficiency.",
            "I failed the embedded systems lab quiz.",
            "The sunset at the airstrip was beautiful."
        ],
        "label": [1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1]
    }
    df = pd.DataFrame(raw_data)
  # shuffling and split
    df_train, df_test = train_test_split(df, test_size=0.2, random_state=seed, shuffle=True)
    return df_train, df_test

Q2. Clean and tokenize text

Implement a basic cleaner: lowercase, strip, replace multiple spaces with one, and remove punctuation
(.,!?;:).
Tokenize by whitespace.
Add a new column tokens that stores a list of tokens per row.
Return the updated DataFrame.

In [41]:
import re

def clean_text(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r'[.,!?;:]', '', s)
    s = re.sub(r'\s+', ' ', s)     # multi spaces to singlee
    return s

def add_tokens_column(df: pd.DataFrame) -> pd.DataFrame:
    df['tokens'] = df['text'].apply(lambda x: clean_text(x).split())
    return df

Q3. Build a vocabulary + token/id mappings

Build token2id and id2token using the training tokens.
Include special tokens: [PAD], [UNK], [BOS], [EOS] at the beginning.
Add tokens that occur at least min_freq times.
Return: token2id (dict), id2token (list).

In [42]:
from collections import Counter
from typing import Dict, List
SPECIALS = ['[PAD]', '[UNK]', '[BOS]', '[EOS]']
def build_vocab(list_of_token_lists, min_freq: int = 1):
  # TODO
  counter = Counter()
  for tokens in list_of_token_lists:
      counter.update(tokens)


  id2token: List[str] = []
  token2id: Dict[str, int] = {}

  for tok in SPECIALS:
      token2id[tok] = len(id2token)
      id2token.append(tok)


  for token, freq in sorted(counter.items()):
      if freq >= min_freq:
          token2id[token] = len(id2token)
          id2token.append(token)

  return token2id, id2token


Q4. Convert tokens to ids + pad to a batch

Implement tokens_to_ids for one sequence.
Implement pad_batch that takes a list of id sequences and returns:
- X: int array (B,T) padded with pad_id
- pad_mask: bool array (B,T) where True means 'real token' and False means padding

In [43]:
import numpy as np
def tokens_to_ids(tokens, token2id, unk_token='[UNK]'):
  # TODO
  unk_id = token2id[unk_token]
  ids = [token2id.get(tok, unk_id) for tok in tokens]
  return ids

def pad_batch(id_seqs, pad_id: int):
  """Return X (B,T) and pad_mask (B,T)"""
  # TODO
  batch_size = len(id_seqs)
  max_len = max(len(seq) for seq in id_seqs)
  X = np.full((batch_size, max_len), pad_id, dtype=np.int64)
  pad_mask = np.zeros((batch_size, max_len), dtype=bool)

  for i, seq in enumerate(id_seqs):
          seq_len = len(seq)
          X[i, :seq_len] = seq
          pad_mask[i, :seq_len] = True
  return X, pad_mask


#Part B - Core Transformer math

Q5. Embedding lookup

Implement an embedding table E of shape (V,D) initialized from a normal distribution (mean 0, std 0.02).
Given token ids X (B,T), return embeddings of shape (B,T,D) using NumPy indexing.


In [44]:
def init_embeddings(vocab_size: int, d_model: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    # Mean = 0 , 
    return rng.normal(0, 0.02, (vocab_size, d_model))

def embed(X: np.ndarray, E: np.ndarray):
    """X: (B,T) int, E: (V,D) -> out: (B,T,D)."""
    return E[X]


Q6. Sinusoidal positional encoding

Implement the classic sinusoidal positional encoding PE of shape (T,D).
Then add it to token embeddings (B,T,D).
Make sure your implementation works for both even and odd D.

In [45]:
def sinusoidal_positional_encoding(T: int, D: int):
    # denominators = 10000^(2i/D)
    indices = np.arange(0, D, 2)
    denom = np.power(10000, indices / D)
    
    positions = np.arange(T)
    # Outer product
    angles = np.outer(positions, 1.0 / denom)
    
    PE = np.zeros((T, D))
    PE[:, 0::2] = np.sin(angles)
    PE[:, 1::2] = np.cos(angles[:, :D//2])
    return PE

def add_positional_encoding(X_emb: np.ndarray, PE: np.ndarray):
    # Slice PE match the batch seq length
    return X_emb + PE[None, :X_emb.shape[1], :]

Q7. Scaled dot-product attention with masking

Implement scaled dot-product attention:
Attention(Q,K,V) = softmax((Q @ K^T) / sqrt(dk) + mask) @ V
Inputs: Q,K,V are (B,H,T,Dh). Mask is boolean broadcastable to (B,H,T,T) where False means 'mask out'.
Requirements:
- Use a numerically stable softmax (subtract max).
- Convert boolean mask to large negative values before softmax.
Return: context (B,H,T,Dh) and attention weights (B,H,T,T).

In [46]:
def softmax(logits: np.ndarray, axis: int = -1):
    shifted = logits - np.max(logits, axis=axis, keepdims=True)
    exp_vals = np.exp(shifted)
    return exp_vals / np.sum(exp_vals, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    dk = Q.shape[-1]
    # Scaled dotproductt
    raw_weights = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(dk)
    
    if mask is not None:
        raw_weights = np.where(mask, raw_weights, -2e9)
        
    attn_map = softmax(raw_weights, axis=-1)
    output = np.matmul(attn_map, V)
    return output, attn_map

Q8. Multi-head self-attention (MHA)

Implement multi-head self-attention for input X (B,T,D).
- Project to Q,K,V using weight matrices Wq,Wk,Wv each (D,D).
- Reshape/split into heads -> (B,H,T,Dh) where Dh=D/H.
- Apply scaled dot-product attention with a pad mask (B,T) (broadcast it appropriately).
- Concatenate heads and apply output projection Wo (D,D).
Return: out (B,T,D) and attention weights (B,H,T,T).

In [47]:
def mha_self_attention(X, Wq, Wk, Wv, Wo, n_heads: int, pad_mask=None):
    B, T, D = X.shape
    dh = D // n_heads
    
    q_all = np.dot(X, Wq).reshape(B, T, n_heads, dh).transpose(0, 2, 1, 3)
    k_all = np.dot(X, Wk).reshape(B, T, n_heads, dh).transpose(0, 2, 1, 3)
    v_all = np.dot(X, Wv).reshape(B, T, n_heads, dh).transpose(0, 2, 1, 3)
    
    # Prep_mask for broadcasting(B,H,T,T)
    m = pad_mask[:, None, None, :] if pad_mask is not None else None
    
    context, weights = scaled_dot_product_attention(q_all, k_all, v_all, m)
   
    merged = context.transpose(0, 2, 1, 3).reshape(B, T, D)
    return np.dot(merged, Wo), weights

Q9. LayerNorm + residual connection

Implement LayerNorm for X (B,T,D) using learnable gamma and beta of shape (D,).
Then implement residual_add_and_norm(Y, X, gamma, beta) that returns LayerNorm(X + Y).

In [48]:
import numpy as np

def layer_normalization(X: np.ndarray, g: np.ndarray, b: np.ndarray, eps: float = 1e-6):
    mu = X.mean(axis=-1, keepdims=True)
    sigma_sq = X.var(axis=-1, keepdims=True)
    
    # Standardize and scale/shift
    X_hat = (X - mu) / np.sqrt(sigma_sq + eps)
    return g * X_hat + b

def residual_add_and_norm(identity: np.ndarray, transformation: np.ndarray, gamma: np.ndarray, beta: np.ndarray):

    return layer_normalization(identity + transformation, gamma, beta)

Q10. Position-wise FeedForward network

Implement FFN: FFN(X) = relu(X @ W1 + b1) @ W2 + b2
Shapes: X (B,T,D), W1 (D,Dff), b1 (Dff,), W2 (Dff,D), b2 (D,)
Return: (B,T,D).

In [49]:
def ffn_block(X: np.ndarray, W_in: np.ndarray, b_in: np.ndarray, W_out: np.ndarray, b_out: np.ndarray):
    # Linear_1+ ReLU
    inner_rep = np.maximum(0, np.dot(X, W_in) + b_in)
    
    # Linear 2
    return np.dot(inner_rep, W_out) + b_out

# Part C - Putting it together

Q11. One Transformer encoder block (forward)

Implement a single encoder block forward pass:
1) MHA = MultiHeadSelfAttention(X) with pad_mask
2) X1 = LayerNorm(X + MHA)
3) FFN = FeedForward(X1)
4) X2 = LayerNorm(X1 + FFN)
Return X2.
You may pass all parameters explicitly (weights, gamma/beta).

In [50]:
def encoder_block_forward(X, params, n_heads: int, pad_mask=None):
    # 1. multi-head self_attention
    attn_out, _ = mha_self_attention(
        X, params['Wq'], params['Wk'], params['Wv'], params['Wo'], 
        n_heads, pad_mask
    )
    
    # 2. first Residual+ norm
    X_mid = residual_add_and_norm(X, attn_out, params['g1'], params['b1'])
    
    # 3. feed_forward Network
    ff_out = ffn_block(
        X_mid, params['W1'], params['b1_ff'], params['W2'], params['b2_ff']
    )
    
    # 4. second residual+ norm(Post_FFN)
    X_final = residual_add_and_norm(X_mid, ff_out, params['g2'], params['b2'])
    
    return X_final


Q12. Sequence classification head + end-to-end demo

Create an end-to-end forward pass for a tiny classifier:
- Input ids -> embeddings + positional enc
- One encoder block
- Pooling: take the [BOS] position (t=0) as the sequence representation
- Linear head: logits = h0 @ Wcls + bcls with Wcls (D,2), bcls (2,)
- Softmax to probabilities
Write predict_proba that takes a batch of texts and returns probs (B,2).
Include simple sanity checks: shapes, probabilities sum to 1, and masking doesn't crash for different
lengths.


In [51]:
def predict_proba(texts, token2id, E, PE, params, W_head, b_head, n_heads: int):
    # Tokenize and add [BOS]
    tokenized = [ ['[BOS]'] + clean_text(t).split() for t in texts ]
    id_sequences = [ tokens_to_ids(toks, token2id) for toks in tokenized ]
    
    # Batching & Padding
    X_ids, mask = pad_batch(id_sequences, token2id['[PAD]'])
    
    # Embeddings + Positional Encoding
    h = embed(X_ids, E)
    # Match sequence length for PE
    h = h + PE[:h.shape[1], :]
    
    # Encoder Block 
    h_encoded = encoder_block_forward(h, params, n_heads, pad_mask=mask)
    
    # Pooling (Extract [BOS] at index 0)
    cls_repr = h_encoded[:, 0, :]
    
    logits = np.dot(cls_repr, W_head) + b_head
    
    exp_logits = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
    probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)
    
    assert probs.shape == (len(texts), 2), f"Expected shape (B, 2), got {probs.shape}"
    assert np.allclose(probs.sum(axis=1), 1.0), "Probabilities do not sum to 1"
    
    return probs